In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

## Env

In [ ]:
from src.utils.env import prepare_environment

_ = prepare_environment("../api_keys")

## Model

In [ ]:
from src.configs import art_configs
from src.configs import vllm_configs
from src.models import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

model_name = "unsloth/Qwen2.5-14B-Instruct"
project_name = "easy2hard_dac"


path_config = PathConfig(
    base_model=model_name,
    project_name=project_name,
)

In [ ]:
import art

host = "0.0.0.0"
port = 8200

model = art.Model(
    name=model_name,
    project=path_config.project_name,
    inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
    inference_base_url=f"https://{host}:{port}/v1",
)

## Data

In [ ]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [ ]:
from src.vllm_client import VllmClient, VllmRouter

inference_clients: list[VllmClient] = []
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=model_name))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [ ]:
from experiments.easy2hard.trainer import Easy2HardTrainer
from src.trainer import PromptConfig, StopCriteria

prompt_config = PromptConfig() # default system prompts
stop_criteria = StopCriteria()

trainer = Easy2HardTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    stop_criteria=stop_criteria,
)

### Inference Test

In [ ]:
import random

idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]
problem = sample["problem"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

await trainer.sync_lora()
preds = await trainer.predict([sample], verbose=True, seed=0)

## Run Benchmark

In [ ]:
trajectories = await trainer.rollout(
    train_data.to_list(),
    max_exceptions=0.025,
)